# Colab · Lab 02b (CNN)

This notebook clones the latest code from `Muhammad-Hamza-Farooq/mlops-lab-2-`, installs dependencies, and runs the Lab 02b CNN notebook.


In [ ]:
# Clone latest repo (safe to re-run)
import os
from pathlib import Path

REPO_URL = "https://github.com/Muhammad-Hamza-Farooq/mlops-lab-2-"
REPO_DIR = Path("mlops-lab-2-")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch --all -p
!git reset --hard origin/master

# We will run the Lab 02b CNN from the lab08 folder
LAB_DIR = Path("lab08")
assert LAB_DIR.exists(), f"Expected {LAB_DIR} to exist. Repo layout changed?"
%cd {LAB_DIR}

print("cwd:", os.getcwd())
print("contents:", sorted([p.name for p in Path('.').iterdir()])[:20])


In [ ]:
# Install dependencies (Colab)
import sys

# Core pinned deps from repo
!{sys.executable} -m pip install -q -r ../requirements/prod.in

# Lab/training deps not included in prod.in
!{sys.executable} -m pip install -q pytorch-lightning==2.1.3 torchmetrics tensorboard wandb scipy toml nltk

# Avoid W&B login prompts
import os
os.environ.setdefault("WANDB_SILENT", "true")
os.environ.setdefault("WANDB_MODE", "disabled")

# Make `text_recognizer` importable
import sys
from pathlib import Path
repo_root = str(Path.cwd())  # currently in .../lab08
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("python:", sys.version)


In [ ]:
# Quick sanity checks: imports + GPU
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# Smoke import
from text_recognizer.data.emnist import EMNIST
from text_recognizer.models.cnn import CNN

print("EMNIST/CNN import ok")


In [ ]:
# Train Lab 02b CNN on EMNIST (M3)
# This runs the same training entrypoint used by the labs.

import sys

cmd = [
    sys.executable,
    "training/run_experiment.py",
    "--data_class",
    "EMNIST",
    "--model_class",
    "CNN",
    "--loss",
    "cross_entropy",
    "--max_epochs",
    "3",
    "--batch_size",
    "256",
    "--num_workers",
    "2",
]

# Use GPU if available (Lightning 2.1 supports these flags)
if torch.cuda.is_available():
    cmd += ["--accelerator", "gpu", "--devices", "1"]
else:
    cmd += ["--accelerator", "cpu", "--devices", "1"]

print("Running:", " ".join(cmd))
import subprocess
subprocess.check_call(cmd)


In [ ]:
# View training logs in TensorBoard
# After training, run this and click the TensorBoard link.

%load_ext tensorboard
%tensorboard --logdir training/logs
